# Jiaozi — Model Soup experiment (Phase 0)

Does a **greedy soup** of K finetuned variants beat the best single model? If yes, we get
ensemble-like accuracy at **single-model inference cost** — the edge under a strict
single-model deployment budget.

**Before running**
1. Select a **GPU** runtime (it trains K times).
2. Add `OPENAI_API_KEY` (and `OPENAI_BASE_URL` if you use a proxy) to Colab **Secrets** —
   Module 1 needs an LLM. Module 4 uses a template (no LLM needed).
3. Run the cells in order. Edit `DATASET / QUERY / N / EPOCHS` in the last cell.

In [ ]:
# Checkout the model-soup branch + install deps
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Isso-W/Jiaozi.git'
REPO_REF = 'feature/model-soup'
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Jiaozi (model-soup) ready at', REPO_DIR)

## Mount Drive (cache dataset + persist checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/Jiaozi/hf_cache'   # dataset downloads once
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
OUTPUT_DIR = '/content/drive/MyDrive/Jiaozi/soup_run'              # project + variants + soup
print('HF cache:', os.environ['HF_HOME'])
print('output  :', OUTPUT_DIR)

## LLM credentials (Module 1 only)

Module 1 parses the query with an LLM. Module 4 uses a validated template (no LLM).

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def _secret(name):
    if userdata is not None:
        try:
            return (userdata.get(name) or '').strip()
        except Exception:
            return ''
    return ''

os.environ['OPENAI_API_KEY'] = _secret('OPENAI_API_KEY') or getpass('OPENAI_API_KEY: ').strip()
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'          # Module 1 -> openai
os.environ['M1_OPENAI_MODEL'] = 'gpt-5.5'
_base = _secret('OPENAI_BASE_URL')
if _base:
    os.environ['OPENAI_BASE_URL'] = _base
print('Module 1 LLM configured (provider=openai, model=gpt-5.5)')

## Run the experiment

Generate a project (Module 1→2→3→4), train **N** seed variants, greedy-soup them, and
print **soup vs best-single**. Edit the knobs below.

In [ ]:
import json
from pathlib import Path

# ── knobs ──────────────────────────────────────────────────────────────────
DATASET = 'dpdl-benchmark/cassava'   # any HuggingFace image dataset
QUERY   = 'Classify cassava leaf images, balancing accuracy and training speed.'
N       = 4                          # number of variants to soup
EPOCHS  = 10                         # epochs per variant
# ───────────────────────────────────────────────────────────────────────────

from pipeline import run_pipeline, parse_dataset_id
import soup

dataset_id, subset = parse_dataset_id(DATASET)
out = Path(OUTPUT_DIR)

print('Step 1/3 — generating project via the pipeline ...')
run_pipeline(QUERY, dataset_id, fmt='nl', subset=subset,
             module4_output=out, module4_real_training=True,
             module4_skip_smoke=True, module4_llm_provider=None)
PROJECT = out / 'module4_code'
cfg = json.loads((PROJECT / 'configs.json').read_text(encoding='utf-8'))[0]
backbone = cfg.get('model_config', cfg).get('backbone') or cfg.get('backbone')
print('Module 3 picked backbone:', backbone)

print(f'\nStep 2/3 — training {N} variants x {EPOCHS} epochs ...')
checkpoints = soup.train_variants(PROJECT, N, EPOCHS, dataset_id)

print('\nStep 3/3 — greedy soup ...')
souped, report = soup.greedy_soup(PROJECT, checkpoints)
soup.save_soup(souped, PROJECT / 'checkpoints' / 'soup_model.pt')

print('\n' + '#' * 60)
print(json.dumps(report, indent=2, ensure_ascii=False))
verdict = 'SOUP WINS' if report['improvement'] > 0 else 'no gain'
print(f"\nbackbone={backbone}  in_soup={report['n_in_soup']}/{N}")
print(f"best_single={report['best_single']:.4f}   soup={report['soup_score']:.4f}   -> {verdict}")
print('#' * 60)